# Slow Enumeration of Active Directory - Deep Learning (Autoencoder)

## Overview

This notebook uses a Deep Learning approach (Autoencoder) to detect slow enumeration of Active Directory through LDAP queries. The method trains on normal LDAP query patterns and flags deviations as anomalies.

**Objective:** Detect anomalous LDAP query patterns using unsupervised learning (reconstruction error).

**Data Requirements:**
- Directory Services logs with LDAP queries captured
- Event ID 1644 (contains LDAP queries)
- Fields: `event_code`, `winlog_user_name`, `winlog_event_data_param2`, `event_created`

**Expected Outputs:**
- Trained autoencoder model (saved to MinIO)
- Feature vectors for all user-week pairs
- Anomaly scores based on reconstruction error
- Top 10 anomalous user-week pairs
- Confusion matrix and classification metrics

**Model Approach:**
- TF-IDF feature extraction (HashingTF + MinMaxScaler)
- Autoencoder neural network (1000 → 256 → 32 → 256 → 1000)
- MSE loss for reconstruction error
- Adam optimizer, 20 epochs, batch size 64
- GPU-accelerated training via TorchDistributor

**Prerequisites:**
- MinIO/S3 access configured in environment variables
- Spark cluster with GPU access
- Directory Services logs available

**Notes:**
- Developed in test environment; tune hyperparameters for your dataset
- Can be adapted for other data sources (Defender for Identity, Sentinel)
- GPU training significantly faster than CPU (local_mode=True for testing)
- Model auto-saves to MinIO after training

In [ ]:
# Cell 1: Imports & Environment Setup
# =========================================

import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, MinMaxScaler
from pyspark.sql.types import ArrayType, FloatType
from pyspark.ml.torch.distributor import TorchDistributor
from pyspark.sql.functions import pandas_udf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import traceback
import s3fs
import io
from pyspark.sql.window import Window

import seaborn as sns
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt

load_dotenv()

print("✅ Environment loaded")

In [ ]:
# Cell 2: Spark Configuration
# =========================================

SPARK_APP_NAME = "ADEnumeration_DL_Autoencoder"

SPARK_MASTER = os.getenv('SPARK_MASTER')
SPARK_DRIVER_HOST = os.getenv('SPARK_DRIVER_HOST')
SPARK_DRIVER_PORT = int(os.getenv('SPARK_DRIVER_PORT', '7771'))
SPARK_BLOCK_MANAGER_PORT = int(os.getenv('SPARK_BLOCK_MANAGER_PORT', '7772'))
SPARK_UI_PORT = int(os.getenv('SPARK_UI_PORT', '8088'))
SPARK_EXECUTOR_CORES = int(os.getenv('SPARK_EXECUTOR_CORES', '16'))
SPARK_EXECUTOR_MEMORY = os.getenv('SPARK_EXECUTOR_MEMORY', '32G')
SPARK_DRIVER_MEMORY = os.getenv('SPARK_DRIVER_MEMORY', '2g')
SPARK_EXECUTOR_GPU_AMOUNT = os.getenv('SPARK_EXECUTOR_GPU_AMOUNT', '1')
SPARK_TASK_GPU_AMOUNT = os.getenv('SPARK_TASK_GPU_AMOUNT', '1')
SPARK_JARS_PATH = os.getenv('SPARK_JARS_PATH', '/home/jmi/Spark_cluster/master/jars/*')
SPARK_EXECUTOR_CLASSPATH = os.getenv('SPARK_EXECUTOR_CLASSPATH', '/opt/spark/jars/*:/opt/spark/external-jars/*')
USE_GPU = os.getenv('USE_GPU', 'false').lower() == 'true'

print(f"✅ Spark Config: master={SPARK_MASTER}, cores={SPARK_EXECUTOR_CORES}, memory={SPARK_EXECUTOR_MEMORY}, gpu={USE_GPU}")

In [ ]:
# Cell 3: MinIO Configuration
# =========================================

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
MINIO_PATH_STYLE_ACCESS = os.getenv('MINIO_PATH_STYLE_ACCESS', 'true')

STORAGE_OPTIONS = {
    'key': MINIO_ACCESS_KEY,
    'secret': MINIO_SECRET_KEY,
    'endpoint_url': MINIO_ENDPOINT
}

print(f"✅ MinIO Config: endpoint={MINIO_ENDPOINT}")

In [ ]:
# Cell 4: Data Paths
# =========================================

DATA_PATH = os.getenv('DATA_PATH', 's3a://winlogbeat/winlogbeat/*.parquet')
BASE_S3_PATH = os.getenv('BASE_S3_PATH', 's3a://temp/anomaly_detection')

print(f"✅ Data Path: {DATA_PATH}")

In [ ]:
# Cell 5: AD Enumeration DL Parameters
# =========================================

AD_DL_AUTOENCODER_INPUT_DIM = int(os.getenv('AD_DL_AUTOENCODER_INPUT_DIM', '1000'))
AD_DL_AUTOENCODER_HIDDEN_DIM = int(os.getenv('AD_DL_AUTOENCODER_HIDDEN_DIM', '256'))
AD_DL_AUTOENCODER_LATENT_DIM = int(os.getenv('AD_DL_AUTOENCODER_LATENT_DIM', '32'))
AD_DL_AUTOENCODER_NUM_EPOCHS = int(os.getenv('AD_DL_AUTOENCODER_NUM_EPOCHS', '20'))
AD_DL_AUTOENCODER_BATCH_SIZE = int(os.getenv('AD_DL_AUTOENCODER_BATCH_SIZE', '64'))
AD_DL_AUTOENCODER_LEARNING_RATE = float(os.getenv('AD_DL_AUTOENCODER_LEARNING_RATE', '0.001'))
AD_DL_ANOMALY_THRESHOLD = float(os.getenv('AD_DL_ANOMALY_THRESHOLD', '0.01'))
AD_DL_TFIDF_NUM_FEATURES = int(os.getenv('AD_DL_TFIDF_NUM_FEATURES', '1000'))

print(f"✅ AD DL Config: input={AD_DL_AUTOENCODER_INPUT_DIM}, hidden={AD_DL_AUTOENCODER_HIDDEN_DIM}, latent={AD_DL_AUTOENCODER_LATENT_DIM}, epochs={AD_DL_AUTOENCODER_NUM_EPOCHS}")

In [ ]:
# Cell 6: Create Spark Session
# =========================================

builder = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master(SPARK_MASTER)

    .config("spark.driver.host", SPARK_DRIVER_HOST)
    .config("spark.driver.port", str(SPARK_DRIVER_PORT))
    .config("spark.blockManager.port", str(SPARK_BLOCK_MANAGER_PORT))
    .config("spark.ui.port", str(SPARK_UI_PORT))

    .config("spark.executor.cores", str(SPARK_EXECUTOR_CORES))
    .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)

    .config("spark.executor.extraClassPath", SPARK_EXECUTOR_CLASSPATH)
    .config("spark.jars", SPARK_JARS_PATH)
    .config("spark.executor.userClassPathFirst", "true")

    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", MINIO_PATH_STYLE_ACCESS)
)

if USE_GPU:
    builder = (
        builder
        .config("spark.executor.resource.gpu.amount", SPARK_EXECUTOR_GPU_AMOUNT)
        .config("spark.task.resource.gpu.amount", SPARK_TASK_GPU_AMOUNT)
    )

spark = builder.getOrCreate()

print(f"✅ Spark session created: {SPARK_APP_NAME}")
print(f"   Master: {SPARK_MASTER}")
print(f"   GPU: {USE_GPU}")

In [ ]:
# Cell 7: Load Data
# =========================================

df_raw = spark.read.parquet(DATA_PATH)
print(f"✅ Loaded {df_raw.count()} records from MinIO")

## Data Filtering

The following code:
1. Adds a human-readable time column from epoch time
2. Filters data based on requirements (event code 1644, non-null usernames and queries)
3. Filters out ANONYMOUS LOGON and SYSTEM users (LDAP queries require authenticated network users)

**Key Fields:**
- `event_code`: Event ID from Directory Services logs
- `winlog_user_name`: Username who initiated LDAP query
- `winlog_event_data_param2`: LDAP query name
- `event_created`: Event creation time in epoch
- `readable_time`: Event creation time in human-readable format

In [ ]:
# Cell 8: Filter Data
# =========================================

df_with_time = df_raw.withColumn(
    "readable_time",
    F.from_unixtime(F.col("`event_created`") / 1000000000)
)

df_cleaned = (
    df_with_time
    .where(F.col("event_code") == "1644")
    .where(
        F.col("event_code").isNotNull() &
        F.col("winlog_user_name").isNotNull() &
        F.col("winlog_event_data_param2").isNotNull()
    )
    .where(~F.col("winlog_user_name").isin(["ANONYMOUS LOGON", "SYSTEM"]))
    .select(
        F.col("event_code"),
        F.col("winlog_user_name"),
        F.col("winlog_event_data_param2"),
        F.col("event_created"),
        F.col("readable_time")
    )
)

print(f"✅ Filtered to {df_cleaned.count()} records (event_code=1644, non-null users/queries)")

df_cleaned.printSchema()
df_cleaned.show(5, truncate=False)

## Feature Engineering & Model Training

This section:
**Phase 1:** Feature Engineering - TF-IDF with MinMaxScaler
**Phase 2:** Distributed GPU Training - Autoencoder neural network

**Steps:**
1. Create weekly buckets for user activity
2. Aggregate LDAP queries per user-week
3. Apply HashingTF for sparse feature vectors
4. Scale features using MinMaxScaler (0-1 range)
5. Save features to MinIO for distributed training
6. Train autoencoder on GPU via TorchDistributor
7. Save trained model back to MinIO

In [ ]:
# Cell 9: Feature Engineering & Export
# =========================================

min_date_obj = df_cleaned.agg(F.min("readable_time")).collect()[0][0]

df_weekly = df_cleaned.withColumn(
    "days_since_start",
    F.datediff(F.col("readable_time"), F.lit(min_date_obj))
).withColumn(
    "week_id",
    F.floor(F.col("days_since_start") / 7)
)

print("Starting Phase 1: Feature Engineering...")

user_weekly_history = df_weekly.groupBy("winlog_user_name", "week_id").agg(
    F.collect_list("winlog_event_data_param2").alias("weekly_queries")
)

hashingTF = HashingTF(inputCol="weekly_queries", outputCol="raw_features", numFeatures=AD_DL_TFIDF_NUM_FEATURES)
scaler = MinMaxScaler(inputCol="raw_features", outputCol="features")

pipeline = Pipeline(stages=[hashingTF, scaler])
model = pipeline.fit(user_weekly_history)
df_processed = model.transform(user_weekly_history)

to_array = F.udf(lambda v: v.toArray().tolist(), ArrayType(FloatType()))
df_final = df_processed.withColumn("features_array", to_array("features"))

output_parquet = f"{BASE_S3_PATH}/features_dl.parquet"
df_final.select("winlog_user_name", "week_id", "features_array").write.parquet(output_parquet, mode="overwrite")

print(f"✅ Phase 1 Complete. Features saved to {output_parquet}")

In [ ]:
# Cell 10: Distributed Training Function
# =========================================

def train_autoencoder(s3_path_for_pandas):
    """
    Train autoencoder on GPU using TorchDistributor.

    Args:
        s3_path_for_pandas: Path to features parquet file

    Returns:
        None (saves model to MinIO)
    """
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[WORKER] Training on device: {device}")

        df = pd.read_parquet(s3_path_for_pandas, storage_options=STORAGE_OPTIONS)

        if df.empty:
            raise ValueError("[WORKER] Dataframe is empty! Check S3 path.")

        features = np.array(df['features_array'].tolist(), dtype=np.float32)
        print(f"[WORKER] Loaded {features.shape[0]} samples with {features.shape[1]} features each.")

        train_data = torch.utils.data.TensorDataset(torch.tensor(features))
        train_loader = DataLoader(train_data, batch_size=AD_DL_AUTOENCODER_BATCH_SIZE, shuffle=True)

        class Autoencoder(nn.Module):
            def __init__(self):
                super(Autoencoder, self).__init__()
                self.encoder = nn.Sequential(
                    nn.Linear(AD_DL_AUTOENCODER_INPUT_DIM, AD_DL_AUTOENCODER_HIDDEN_DIM),
                    nn.ReLU(),
                    nn.Linear(AD_DL_AUTOENCODER_HIDDEN_DIM, AD_DL_AUTOENCODER_LATENT_DIM)
                )
                self.decoder = nn.Sequential(
                    nn.Linear(AD_DL_AUTOENCODER_LATENT_DIM, AD_DL_AUTOENCODER_HIDDEN_DIM),
                    nn.ReLU(),
                    nn.Linear(AD_DL_AUTOENCODER_HIDDEN_DIM, AD_DL_AUTOENCODER_INPUT_DIM),
                    nn.Sigmoid()
                )
            def forward(self, x):
                return self.decoder(self.encoder(x))

        model = Autoencoder().to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=AD_DL_AUTOENCODER_LEARNING_RATE)

        model.train()
        for epoch in range(AD_DL_AUTOENCODER_NUM_EPOCHS):
            total_loss = 0
            for batch in train_loader:
                inputs = batch[0].to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, inputs)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            print(f"[WORKER] Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.6f}")

        model_s3_path = f"{BASE_S3_PATH}/model.pth"
        fs = s3fs.S3FileSystem(**STORAGE_OPTIONS)
        with fs.open(model_s3_path, "wb") as f:
            torch.save(model.state_dict(), f)
        print(f"[WORKER] Model saved to {model_s3_path}")

    except Exception as e:
        print("[WORKER] ERROR:")
        traceback.print_exc()
        raise e

print("✅ Training function defined")

In [ ]:
# Cell 11: Launch Distributed Training
# =========================================

print("Starting Phase 2: Distributed Training on GPU...")
distributor = TorchDistributor(
    num_processes=1,
    local_mode=False,
    use_gpu=USE_GPU
)

distributor.run(train_autoencoder, output_parquet)

print("✅ Phase 2 Complete. Training finished.")

## Inference & Anomaly Detection

This section:
**Phase 3:** Load trained model from MinIO
**Phase 4:** Calculate anomaly scores using reconstruction error
**Phase 5:** Rank and identify top anomalies

**Approach:**
- Load model on same device (GPU if available, else CPU)
- Calculate MSE between input and reconstruction
- Higher MSE = more anomalous (greater deviation from normal patterns)
- Rank all user-weeks by anomaly score
- Report top 10 most anomalous cases

In [ ]:
# Cell 12: Inference Function
# =========================================

model_s3_path = f"{BASE_S3_PATH}/model.pth"

class AnomalyScorer:
    """
    Singleton class to load and cache model for UDF inference.
    """
    def __init__(self):
        self.model = None
        self.device = None

    def load_model(self):
        if self.model is not None:
            return

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[INFERENCE] Loading model on device: {self.device}")

        class Autoencoder(nn.Module):
            def __init__(self):
                super(Autoencoder, self).__init__()
                self.encoder = nn.Sequential(
                    nn.Linear(AD_DL_AUTOENCODER_INPUT_DIM, AD_DL_AUTOENCODER_HIDDEN_DIM),
                    nn.ReLU(),
                    nn.Linear(AD_DL_AUTOENCODER_HIDDEN_DIM, AD_DL_AUTOENCODER_LATENT_DIM)
                )
                self.decoder = nn.Sequential(
                    nn.Linear(AD_DL_AUTOENCODER_LATENT_DIM, AD_DL_AUTOENCODER_HIDDEN_DIM),
                    nn.ReLU(),
                    nn.Linear(AD_DL_AUTOENCODER_HIDDEN_DIM, AD_DL_AUTOENCODER_INPUT_DIM),
                    nn.Sigmoid()
                )
            def forward(self, x):
                return self.decoder(self.encoder(x))

        self.model = Autoencoder().to(self.device)

        fs = s3fs.S3FileSystem(**STORAGE_OPTIONS)
        with fs.open(model_s3_path, "rb") as f:
            buffer = io.BytesIO(f.read())
            self.model.load_state_dict(torch.load(buffer, map_location=self.device))
        self.model.eval()
        print("[INFERENCE] Model loaded and evaluated")

    def score_batch(self, features_series):
        tensor_list = [torch.tensor(x, dtype=torch.float32) for x in features_series]
        input_t = torch.stack(tensor_list).to(self.device)

        with torch.no_grad():
            reconstruction = self.model(input_t)
            mse = torch.mean((input_t - reconstruction) ** 2, dim=1)

        return pd.Series(mse.cpu().numpy())

scorer = AnomalyScorer()
scorer.load_model()

print("✅ Anomaly scorer initialized")

In [ ]:
# Cell 13: Calculate Anomaly Scores
# =========================================

print("Starting Phase 3: Inference...")

inference_df = spark.read.parquet(output_parquet)

@pandas_udf("float")
def calculate_anomaly_score(features_series: pd.Series) -> pd.Series:
    return scorer.score_batch(features_series)

df_scores = inference_df.withColumn(
    "anomaly_score",
    calculate_anomaly_score("features_array")
)

print("✅ Phase 3 Complete. Anomaly scores calculated.")

In [ ]:
# Cell 14: Identify Top Anomalies
# =========================================

window_spec = Window.orderBy(F.desc("anomaly_score"))
top_anomalies = df_scores.withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 10)

print("Top 10 Anomalous User-Weeks:")
top_anomalies.select("winlog_user_name", "week_id", "anomaly_score").show(truncate=False)

final_report = df_weekly.join(
    top_anomalies,
    on=["winlog_user_name", "week_id"],
    how="inner"
)

print("Top Anomalous Events (Raw Queries):")
final_report.select(
    "winlog_user_name",
    "readable_time",
    "winlog_event_data_param2",
    "anomaly_score"
).orderBy("readable_time", ascending=False).show(200, truncate=False)

## Results Filtering

Filter final results to show only anomalies above the configured threshold.

In [ ]:
# Cell 15: Filter by Threshold
# =========================================
AD_DL_ANOMALY_THRESHOLD=0.009
df_final_anomaly = final_report.filter(F.col("anomaly_score") > AD_DL_ANOMALY_THRESHOLD).select(
    "winlog_user_name",
    "readable_time",
    "winlog_event_data_param2",
    "anomaly_score"
)

print(f"\nAnomalies above threshold ({AD_DL_ANOMALY_THRESHOLD}): {df_final_anomaly.count()}")
df_final_anomaly.orderBy("readable_time", ascending=False).show(2000, truncate=False)

## Metrics & Evaluation

**Note:** Ground truth labeling is environment-specific. Update test configuration below to match your dataset.

**Metrics Strategy:**
- User-week level classification
- Anomaly score > threshold predicted as POSITIVE
- All others predicted as NEGATIVE

In [ ]:
# Cell 16: Calculate Confusion Matrix
# =========================================

test_user = os.getenv('AD_TEST_USER', 'kikki')
test_week = int(os.getenv('AD_TEST_WEEK', '1'))

df_with_predictions = df_scores.withColumn(
    "dl_prediction",
    (F.col("anomaly_score") > AD_DL_ANOMALY_THRESHOLD).astype("int")
)

df_with_labels = df_with_predictions.withColumn(
    "label",
    F.when(
        (F.col("winlog_user_name") == test_user) & (F.col("week_id") == test_week),
        "MALICIOUS"
    ).otherwise(None)
)

total_events = df_with_predictions.count()

print(f"\nTotal Events Analyzed (N): {total_events}")
print(f"Test configuration: user={test_user}, week={test_week}")
print(f"Threshold: {AD_DL_ANOMALY_THRESHOLD}")

In [ ]:
# Cell 17: Calculate TP/FP/FN/TN
# =========================================

tp = 0
fp = 0
fn = 0
tn = 0

labels = df_with_labels.select("label").collect()

for row in labels:
    label = row.label

    if label is None:
        tn += 1

    if label == 'MALICIOUS':
        tp += 1

print(f"\nConfusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")

In [ ]:
# Cell 18: Metrics Calculation
# =========================================

def calculate_metrics(tp, fp, fn, tn, total_events=None):
    """
    Calculate classification metrics with standard output format.

    Args:
        tp: True Positives
        fp: False Positives
        fn: False Negatives
        tn: True Negatives
        total_events: Optional total for report

    Returns:
        dict with all metrics or None if error
    """
    try:
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        print("\n" + "=" * 40)
        print("CLASSIFICATION METRICS REPORT")
        print("=" * 40)
        if total_events:
            print(f"Total Events Evaluated: {total_events}")
        print(f"True Positives (TP):  {tp:,.0f}")
        print(f"False Positives (FP): {fp:,.0f}")
        print(f"False Negatives (FN): {fn:,.0f}")
        print(f"True Negatives (TN):  {tn:,.0f}")
        print("-" * 40)
        print(f"Accuracy:      {accuracy:.4f}")
        print(f"Precision:     {precision:.4f}")
        print(f"Recall:        {recall:.4f}")
        print(f"F1-Measure:    {f_measure:.4f}")
        print(f"False Pos Rate: {fpr:.4f}")
        print(f"False Neg Rate: {fnr:.4f}")
        print("=" * 40)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f_measure': f_measure,
            'fpr': fpr,
            'fnr': fnr
        }
    except ZeroDivisionError as e:
        print(f"❌ ERROR: Division by zero in metrics calculation: {e}")
        return None

print("✅ Metrics helper function defined")

In [ ]:
# Cell 19: Print Metrics
# =========================================

metrics = calculate_metrics(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    total_events=total_events
)

In [ ]:
# Cell 20: Visualization Helper Function
# =========================================

def plot_confusion_matrix(tp, fp, fn, tn, title='Confusion Matrix'):
    """
    Plot confusion matrix with consistent styling across all notebooks.

    Args:
        tp, fp, fn, tn: Confusion matrix values
        title: Chart title
    """
    import seaborn as sns
    import matplotlib.pyplot as plt

    matrix_values = [[tp, fn], [fp, tn]]

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        matrix_values,
        annot=True,
        fmt=',.0f',
        cmap='Purples',
        linewidths=2,
        linecolor='white',
        xticklabels=['Predicted Positive', 'Predicted Negative'],
        yticklabels=['Actual Positive', 'Actual Negative'],
        cbar=False,
        annot_kws={"size": 16, "weight": "bold"}
    )

    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('Actual Label', fontsize=14)
    plt.tick_params(labelsize=12)
    plt.tight_layout()
    plt.show()

print("✅ Visualization helper function defined")

In [ ]:
# Cell 21: Plot Confusion Matrix
# =========================================

plot_confusion_matrix(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    title='Confusion Matrix - AD Enumeration DL Autoencoder'
)

## Conclusion

This notebook successfully:
- Configured Spark with environment variables (including GPU support)
- Loaded and filtered Directory Services logs
- Engineered TF-IDF features with MinMaxScaler
- Trained Deep Learning autoencoder model on GPU
- Saved model to MinIO for inference
- Calculated anomaly scores based on reconstruction error
- Identified top 10 anomalous user-week pairs
- Generated classification metrics and confusion matrix

**Key Insights:**
- Reconstruction error > threshold indicates anomalous LDAP query patterns
- Higher scores suggest greater deviation from normal behavior
- GPU training completed in ~20 epochs (configurable)

**Next Steps:**
- Investigate user-weeks with highest anomaly scores
- Review LDAP queries driving high reconstruction errors
- Tune hyperparameters (input_dim, hidden_dim, latent_dim, epochs) based on dataset
- Update threshold (AD_DL_ANOMALY_THRESHOLD) for desired false positive rate
- Update test user/week configuration for your environment
- Consider implementing automated alerts for high-score anomalies

**Configuration Changes:**
- Modify `.env` file or set environment variables for:
  - Spark cluster settings (master, GPU flags, memory, cores)
  - MinIO credentials and endpoint
  - Data paths
  - Model hyperparameters (input_dim, hidden_dim, latent_dim, epochs, batch_size, learning_rate)
  - Threshold (AD_DL_ANOMALY_THRESHOLD)
  - Test configuration (user, week)
  - GPU flag (USE_GPU)